In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Data load karna
df=pd.read_csv("/content/airlines_flights_data.csv")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/airlines_flights_data.csv'

In [ ]:
df.info()

In [ ]:
df.shape


In [ ]:
df.isna().sum()

In [ ]:
# check the duplicate rows in the data
df.duplicated().sum()

In [ ]:
# drop the index column of the data
df=df.drop('index',axis=1)
df


In [ ]:
# Clean data ko nayi file mein save karein
df.to_csv('cleaned_flights_data.csv', index=False)

In [ ]:
df.shape

In [ ]:
# streamlit ka use karke dashboard reday karte hai
# first pip install karna hai jaha mai jaruri libraries install kar saku
!pip install streamlit plotly


In [ ]:
%%writefile flight.Py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Flight Data Dashboard", layout="wide")
st.title("✈️ Airlines Flight Price Dashboard")

# Welcome Message
st.write("👋 Hello Khushboo! Aapke khud ke banaye hue Flight Dashboard mein aapka swagat hai. Happy Analyzing!")

@st.cache_data
def load_data():
    df = pd.read_csv('airlines_flights_data.csv')
    if 'index' in df.columns:
        df = df.drop('index', axis=1)
    return df.drop_duplicates()

df = load_data()

# --- FILTERS (Sidebar) ---
st.sidebar.header("🔍 Apni Flight Dhundhein")

airlines = ["Sabhi"] + list(df['airline'].unique())
sources = ["Sabhi"] + list(df['source_city'].unique())
destinations = ["Sabhi"] + list(df['destination_city'].unique())

selected_airline = st.sidebar.selectbox("Airline chunein:", airlines)
selected_source = st.sidebar.selectbox("Kahan se (Source):", sources)
selected_dest = st.sidebar.selectbox("Kahan tak (Destination):", destinations)

filtered_df = df.copy()
if selected_airline != "Sabhi":
    filtered_df = filtered_df[filtered_df['airline'] == selected_airline]
if selected_source != "Sabhi":
    filtered_df = filtered_df[filtered_df['source_city'] == selected_source]
if selected_dest != "Sabhi":
    filtered_df = filtered_df[filtered_df['destination_city'] == selected_dest]

# --- TOP METRICS ---
col1, col2, col3 = st.columns(3)
col1.metric(label="Total Flights Found", value=f"{len(filtered_df):,}")
if not filtered_df.empty:
    col2.metric(label="Avg Price", value=f"₹ {round(filtered_df['price'].mean(), 2):,}")
else:
    col2.metric(label="Avg Price", value="₹ 0")
col3.metric(label="Total Airlines", value=filtered_df['airline'].nunique())

st.write("---")

# --- DATA TABLE ---
st.subheader("📋 Flight Data Table")
if not filtered_df.empty:
    st.table(filtered_df.head(50))
else:
    st.warning("Aapke chune gaye filters ke hisaab se koi flight nahi mili.")

st.write("---")

# --- CHARTS ---
if not filtered_df.empty:
    col_left, col_right = st.columns(2)
    with col_left:
        st.subheader("1. Har Airline ki Average Price")
        avg_price = filtered_df.groupby('airline')['price'].mean().reset_index()
        fig1 = px.bar(avg_price, x='airline', y='price', color='airline')
        st.plotly_chart(fig1, use_container_width=True)

    with col_right:
        st.subheader("2. Economy vs Business Class")
        fig2 = px.pie(filtered_df, names='class', color='class', hole=0.4)
        st.plotly_chart(fig2, use_container_width=True)

# --- PRICE TREND ---
st.write("---")
st.subheader("📈 Flight Booking: Days Left vs Price")
if not filtered_df.empty:
    trend_data = filtered_df.groupby(['days_left', 'class'])['price'].mean().reset_index()
    fig3 = px.line(trend_data, x='days_left', y='price', color='class', markers=True)
    fig3.update_xaxes(autorange="reversed")
    st.plotly_chart(fig3, use_container_width=True)

# --- DOWNLOAD BUTTON ---
st.write("---")
st.subheader("💾 Data Download Karein")
if not filtered_df.empty:
    csv = filtered_df.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Filtered Data ko CSV mein Download karein",
        data=csv,
        file_name='filtered_flight_data.csv',
        mime='text/csv',
    )

# --- FLIGHT BOOKING SECTION ---
st.write("---")
st.subheader("🎟️ Flight Ticket Book Karein")
st.markdown("Upar table mein apni pasand ki flight dekhne ke baad yahan se uski ticket book karein.")

with st.form("booking_form", clear_on_submit=True):
    st.write("Apni Details Bharein:")

    col_b1, col_b2 = st.columns(2)
    name = col_b1.text_input("Aapka Poora Naam:")
    phone = col_b2.text_input("Mobile Number:")

    col_b3, col_b4 = st.columns(2)
    passengers = col_b3.number_input("Kitne log safar kar rahe hain?", min_value=1, max_value=10, value=1)
    flight_date = col_b4.date_input("Udne ki Tareekh (Date):")

    submit_button = st.form_submit_button(label="✅ Ticket Book Karein")

    if submit_button:
        if name == "" or phone == "":
            st.error("⚠️ Kripya apna naam aur mobile number zaroor bharein.")
        else:
            st.success(f"🎉 Badhai ho {name}! Aapki {passengers} logo ki flight successfully book ho gayi hai. Ticket ki details aapke number ({phone}) par bhej di jayengi.")
            st.balloons()

In [ ]:
import urllib
print("Dashboard dekhne ke liye Password (Endpoint IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Yahan file ka naam flight.py kar diya gaya hai aur -y lagaya hai
!streamlit run flight.py & npx -y localtunnel --port 8501